# Testing Pipelining Functionality

In [1]:
from src.neuroanalyst.models.about import About
from src.neuroanalyst.models.process.logic.code.python import (
    PythonEncoder, 
    PythonDecoder, 
    PythonEncoderConfig, 
    PythonDecoderConfig,
    encode_logic,
    decode_from_string
)
from src.neuroanalyst.models.process.logic.core import NeuProcessLogic, NeuProcessLogicArgument, ProgrammingLanguage
from src.neuroanalyst.models.process.logic.code.base import CodeGenerationResult
from src.neuroanalyst.models.process.logic.code.python.decoder import PythonDecoder
from src.neuroanalyst.models.process.dir.core import NeuProcessDir, NeuProcessDirConfig
from src.neuroanalyst.models.process.process.core import NeuProcess
from src.neuroanalyst.models.process.exec.core import NeuProcessExec, HPCScheduler
from src.neuroanalyst.models.pipeline.core import NeuPipeline, NeuPipelineStep

import json

## T1w pre-processing pipeline using FSL
### Plan
1. Create processes for individual steps-
    - BET
    - FAST Segmentation
    - Thresholding
2. Create pipeline with runtime parameters

In [ ]:
BIDS_ROOT: str = "/Users/cmokashi/neuroanalyst/datasets/ds004884-1.0.2"
FSL_IMG: str = "/risapps/apptainer/repo/fsl/3.16.8/fsl_3.16.8.sif"

### Load NeuProcessLogic instances

In [2]:
decoder = PythonDecoder()

functions: dict = {
    "fsl_bet": {
        "about": About(
            name="FSL BET",
            description="Brain Extraction Tool (BET) from FSL",
            version="1.0.0",
            author="Chinmay Mokashi"
        ),
        # "file_path": "./src/neuroanalyst/models/process/logic/code/python/samples/fsl_bet.py",
        "file_path": "./src/neuroanalyst/models/process/logic/code/python/samples/fake_fsl_bet.py",
    },
    "fsl_fast": {
        "about": About(
            name="FSL FAST",
            description="FSL FAST Segmentation",
            version="1.0.0",
            author="Chinmay Mokashi"
        ),
        # "file_path": "./src/neuroanalyst/models/process/logic/code/python/samples/fsl_fast.py",
        "file_path": "./src/neuroanalyst/models/process/logic/code/python/samples/fake_fsl_fast.py",
    },
    "fsl_threshold": {
        "about": About(
            name="FSL Threshold",
            description="FSL Thresholding",
            version="1.0.0",
            author="Chinmay Mokashi"
        ),
        "file_path": "./src/neuroanalyst/models/process/logic/code/python/samples/fake_fsl_threshold.py",
    },
}

bet_logic: NeuProcessLogic = decoder.decode_from_file(functions["fsl_bet"]["file_path"])
fast_logic: NeuProcessLogic = decoder.decode_from_file(functions["fsl_fast"]["file_path"])
threshold_logic: NeuProcessLogic = decoder.decode_from_file(functions["fsl_threshold"]["file_path"])

# bet_logic.about = functions["fsl_bet"]["about"]
# fast_logic.about = functions["fsl_fast"]["about"]
# threshold_logic.about = functions["fsl_threshold"]["about"]

### Create NeuProcessDir and respective directories

In [3]:
bet_dir: NeuProcessDir = NeuProcessDir.from_logic(bet_logic)
fast_dir: NeuProcessDir = NeuProcessDir.from_logic(fast_logic)
threshold_dir: NeuProcessDir = NeuProcessDir.from_logic(threshold_logic)

for dir in [bet_dir, fast_dir, threshold_dir]:
    dir.add_environment_variables("FSL_IMG")
    dir.add_language_packages("python", ["nibabel", "numpy"])

bet_dir.generate()
fast_dir.generate()
threshold_dir.generate()

PosixPath('/Users/cmokashi/neuroanalyst/working_dirs/PR-398764')

### Create NeuProcess instances (with virtual environments)

In [ ]:
# bet_process: NeuProcess = NeuProcess.from_process_id(bet_dir.process_id)
# fast_process: NeuProcess = NeuProcess.from_process_id(fast_dir.process_id)
# threshold_process: NeuProcess = NeuProcess.from_process_id(threshold_dir.process_id)

# bet_process.create_virtual_env()
# fast_process.create_virtual_env()
# threshold_process.create_virtual_env()

### Create NeuProcess instances (with Singularity images)


In [ ]:
bet_process: NeuProcess = NeuProcess.from_process_id(bet_dir.process_id)
fast_process: NeuProcess = NeuProcess.from_process_id(fast_dir.process_id)
threshold_process: NeuProcess = NeuProcess.from_process_id(threshold_dir.process_id)

bet_process.build_image()
fast_process.build_image()
threshold_process.build_image()

### Create NeuProcessExec instances

In [ ]:
dataset_path: str = BIDS_ROOT
fsl_img_path: str = FSL_IMG

bet_exec: NeuProcessExec = NeuProcessExec(
    process=bet_process,
    execution_mode="venv",
    env_var_values={"FSL_IMG": fsl_img_path, "BIDS_FILTERS": json.dumps({
        "suffix": "T1w",
        "extension": ".nii.gz"
    })},
    bind_path_values={"/data": dataset_path}
)

fast_exec: NeuProcessExec = NeuProcessExec(
    process=fast_process,
    execution_mode="venv",
    env_var_values={"FSL_IMG": fsl_img_path, "BIDS_FILTERS": json.dumps({
        "desc": "brain",
        "suffix": "T1w",
        "extension": ".nii.gz"
    })},
    bind_path_values={"/data": dataset_path}
)

threshold_exec: NeuProcessExec = NeuProcessExec(
    process=threshold_process,
    execution_mode="venv",
    env_var_values={"FSL_IMG": fsl_img_path, "BIDS_FILTERS": json.dumps({
        "desc": "seg",
        "suffix": "T1w",
        "extension": ".nii.gz"
    })},
    bind_path_values={"/data": dataset_path}
)

# bet_exec.get_configuration_status()
# fast_exec.get_configuration_status()
# threshold_exec.get_configuration_status()

threshold_exec.print_configuration_status()



=== Configuration Status for NeuProcessExec PE-333868 ===

Bind Paths:
  Required (1): /data
  Provided (1): /data
  Missing (0): None

Environment Variables:
  Required (6): BIDS_FILTERS, PIPELINE_ID, PIPELINE_NAME, PROCESS_ID, PROCESS_EXEC_ID, FSL_IMG
  Provided (2): FSL_IMG, BIDS_FILTERS
  Missing (4): PIPELINE_ID, PIPELINE_NAME, PROCESS_ID, PROCESS_EXEC_ID

Execution Command:
  Status: ❌ Command is not set

Overall Status:
  Configuration: ❌ Missing required configuration values
=== End of Configuration Status ===



### Prepare NeuPipeline

In [6]:
scheduler: HPCScheduler = HPCScheduler.LSF
# scheduler: HPCScheduler = HPCScheduler.LOCAL

bet_step: NeuPipelineStep = NeuPipelineStep(
    name="Brain Extraction",
    description="Perform brain extraction using FSL BET",
    process_execs=[bet_exec]
)
fast_step: NeuPipelineStep = NeuPipelineStep(
    name="Tissue Segmentation",
    description="Perform tissue segmentation using FSL FAST",
    process_execs=[fast_exec]
)
threshold_step: NeuPipelineStep = NeuPipelineStep(
    name="Thresholding",
    description="Apply thresholding using FSL Threshold",
    process_execs=[threshold_exec]
)
about_fsl_pipeline: About = About(
    name="T1w Preprocessing Pipeline using FSL",
    description="A pipeline for preprocessing T1-weighted MRI images using FSL tools.",
    version="1.0.0",
    author="Chinmay Mokashi"
)
fsl_pipeline: NeuPipeline = NeuPipeline(
    about=about_fsl_pipeline,
    steps=[bet_step, fast_step, threshold_step],
    scheduler=scheduler
)
fsl_pipeline.apply_standard_exec_params()
fsl_pipeline.steps[2].process_execs[0].print_configuration_status()


=== Configuration Status for NeuProcessExec PE-333868 ===

Bind Paths:
  Required (1): /data
  Provided (1): /data
  Missing (0): None

Environment Variables:
  Required (6): BIDS_FILTERS, PIPELINE_ID, PIPELINE_NAME, PROCESS_ID, PROCESS_EXEC_ID, FSL_IMG
  Provided (6): FSL_IMG, BIDS_FILTERS, PIPELINE_NAME, PIPELINE_ID, PROCESS_ID, PROCESS_EXEC_ID
  Missing (0): None

Execution Command:
  Status: ✅ Command is set
  Command: bsub < /Users/cmokashi/neuroanalyst/working_dirs/PR-398764/execute/hpc/lsf/run_venv.sh --bind /data=/Users/cmokashi/neuroanalyst/datasets/ds004884-1.0.2 --env FSL_IMG=/usr/local/fsl --env BIDS_FILTERS={"desc": "seg", "suffix": "T1w", "extension": ".nii.gz"} --env PIPELINE_NAME=T1w Preprocessing Pipeline using FSL --env PIPELINE_ID=PL-420509 --env PROCESS_ID=PR-398764 --env PROCESS_EXEC_ID=PE-333868

Overall Status:
  Configuration: ✅ All required configuration values are provided
=== End of Configuration Status ===



In [7]:
# fsl_pipeline.steps[0].process_execs[0].generate_command()
fsl_pipeline.create_pipeline_dir()

PosixPath('/Users/cmokashi/neuroanalyst/pipelines/PL-420509')